## Setup

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import requests
import pandas as pd
import pyspark
from io import BytesIO
from dotenv import load_dotenv
from pyspark.sql import functions as func

In [5]:
load_dotenv("etl/.env")

JDBC_PATH = os.getenv("JDBC_PATH")
_DB_URL = os.getenv("DB_URL")
_DB_NAME = os.getenv("DB_NAME")
_DB_USER = os.getenv("DB_USER")
_DB_PASSWORD = os.getenv("DB_PASSWORD")

if JDBC_PATH is None or _DB_URL is None or _DB_NAME is None or _DB_USER is None or _DB_PASSWORD is None:
    raise EnvironmentError("Set .env vars!")
if not os.path.exists(JDBC_PATH) or not os.path.isfile(JDBC_PATH):
    raise FileNotFoundError("JDBC driver not found!")

DB_URL = f"jdbc:mysql://{_DB_URL}/{_DB_NAME}"
DB_PROPERTIES = {"user": _DB_USER, "password": _DB_PASSWORD, "driver": "com.mysql.jdbc.Driver"}

In [6]:
spark = pyspark.sql.SparkSession.builder.appName("ETL TCC").config("spark.jars", JDBC_PATH).getOrCreate()

26/04/21 17:27:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Extract

### Base bueiros

In [ ]:
BUEIROS_URL = "https://www.der.sp.gov.br/WebSite/Arquivos/DadosAbertos/AtivosRodoviarios/Bueiros/Bueiros.xlsx"
res = requests.get(BUEIROS_URL)
res.raise_for_status()

In [ ]:
content_wrapper = BytesIO(res.content)
pandas_df = pd.read_excel(content_wrapper)
df_bueiros_raw = spark.createDataFrame(pandas_df)

### OpenMeteo

### Alagamentos CGE

### Mocks

In [ ]:
pandas_df = pd.read_csv("./mocks/distritos.csv")
df_mock_distritos = spark.createDataFrame(pandas_df)

## Transform

#### 1. Distrito

In [ ]:
df_transf_distritos = df_mock_distritos\
    .withColumn("id", func.monotonically_increasing_id()).withColumn("cidade", func.lit("São Paulo"))

In [ ]:
teste = df_transf_distritos.toPandas()

In [ ]:
teste

,bairro,id,cidade
0,Grajaú,0,São Paulo
1,Jardim Ângela,1,São Paulo
2,Capão Redondo,2,São Paulo
3,Sapopemba,3,São Paulo
4,Sacomã,4,São Paulo
...,...,...,...
91,Barra Funda,94489280515,São Paulo
92,Jaguara,94489280516,São Paulo
93,Sé,94489280517,São Paulo
94,Pari,94489280518,São Paulo


## Load

### 1. Distrito

In [ ]:
df_transf_distritos.write.jdbc(DB_URL, "distrito", properties=DB_PROPERTIES, mode="append")